# HOM Temporal Validation Plots

This notebook is plotting-only. It reads CSV files produced by `run_hom_three_overlap_batch.py` and does not run the simulation.

Default dataset: `results/hom_config`, generated from `configs/hom_config.ini`.

In [ ]:
from pathlib import Path
import configparser

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'sequence').is_dir() and (path / 'example').is_dir():
            return path
    raise RuntimeError('Could not find repo root. Run this notebook from inside the repository.')

REPO_ROOT = find_repo_root()
EXAMPLE_DIR = REPO_ROOT / 'example' / 'hom_swapping_validation'

CONFIG_NAME = 'hom_config'
CONFIG_PATH = EXAMPLE_DIR / 'configs' / f'{CONFIG_NAME}.ini'
RESULTS_DIR = EXAMPLE_DIR / 'results' / CONFIG_NAME

print('Config:', CONFIG_PATH.relative_to(REPO_ROOT))
print('Results:', RESULTS_DIR.relative_to(REPO_ROOT))


In [ ]:
config = configparser.ConfigParser(inline_comment_prefixes=(';', '#'))
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Missing config file: {CONFIG_PATH}')
config.read_string(CONFIG_PATH.read_text(encoding='utf-8-sig'))

def cfg_float(section: str, key: str, default: float) -> float:
    return config.getfloat(section, key, fallback=default)

def load_csv(name: str) -> pd.DataFrame:
    path = RESULTS_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run the batch script first for CONFIG_NAME={CONFIG_NAME!r}.')
    return pd.read_csv(path)

runs = load_csv('hom_temporal_runs.csv')
summary = load_csv('hom_temporal_delay_scan_validation.csv')

DURATION_S = cfg_float('batch', 'duration_s', 1.0)
RUNS_PER_POINT = config.getint('batch', 'runs_per_point', fallback=int(runs['run_index'].nunique()))
SOURCE_FREQUENCY_HZ = cfg_float('source', 'source_frequency_hz', float('nan'))

print(f'raw runs: {len(runs)}')
print(f'delay points: {runs["delay_ps"].nunique()}')
print(f'runs per point in config: {RUNS_PER_POINT}')
print(f'simulated duration per run: {DURATION_S:g} s')
print(f'source frequency: {SOURCE_FREQUENCY_HZ:g} Hz')
summary.head()


In [ ]:
# Convert raw per-run counts to counts/s so error bars are computed across Monte Carlo runs.
count_columns = [
    'bsm1_count',
    'bsm2_count',
    'heraldA_count',
    'heraldB_count',
    'bsm_twofold_count',
    'hom_fourfold_count',
    'selected_herald_fourfold_count',
    'matched_herald_fourfold_count',
    'mismatched_herald_fourfold_count',
    'matched_projected_coincidence_count',
    'mismatched_projected_coincidence_count',
    'interfering_coincidence_count',
]

for col in count_columns:
    if col in runs.columns:
        runs[f'{col}_per_s'] = runs[col] / DURATION_S

rate_columns = [f'{col}_per_s' for col in count_columns if f'{col}_per_s' in runs.columns]

agg = runs.groupby('delay_ps')[rate_columns].agg(['mean', 'std', 'count']).reset_index()
agg.columns = ['_'.join([str(x) for x in col if str(x)]) for col in agg.columns.to_flat_index()]
agg = agg.sort_values('delay_ps').reset_index(drop=True)

def y_mean(col: str) -> np.ndarray:
    return agg[f'{col}_per_s_mean'].to_numpy(float)

def y_std(col: str) -> np.ndarray:
    return agg[f'{col}_per_s_std'].fillna(0).to_numpy(float)

delay = agg['delay_ps'].to_numpy(float)
agg.head()


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(delay, y_mean('bsm_twofold_count'), yerr=y_std('bsm_twofold_count'), marker='o', ms=3, lw=1.5, capsize=2, label='BSM twofold')
ax.errorbar(delay, y_mean('hom_fourfold_count'), yerr=y_std('hom_fourfold_count'), marker='s', ms=3, lw=1.5, capsize=2, label='All heralded fourfold')
ax.set_xlabel('Relative delay (ps)')
ax.set_ylabel('Counts/s')
ax.set_title('BSM twofold and heralded fourfold counts')
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(delay, y_mean('matched_herald_fourfold_count'), yerr=y_std('matched_herald_fourfold_count'), marker='o', ms=3, lw=1.5, capsize=2, label='matched')
ax.errorbar(delay, y_mean('mismatched_herald_fourfold_count'), yerr=y_std('mismatched_herald_fourfold_count'), marker='s', ms=3, lw=1.5, capsize=2, label='mismatched')
ax.set_xlabel('Relative delay (ps)')
ax.set_ylabel('Counts/s')
ax.set_title('Herald-conditioned HOM counts')
ax.legend()
plt.show()


In [ ]:
# Visibility estimate from measured matched fourfold counts/s.
# Baseline is estimated from the outer 20% of absolute delay values; dip minimum is the point nearest zero delay.
matched = y_mean('matched_herald_fourfold_count')
abs_delay = np.abs(delay)
edge_mask = abs_delay >= 0.8 * abs_delay.max()
zero_idx = int(np.argmin(abs_delay))

baseline = float(np.mean(matched[edge_mask]))
dip = float(matched[zero_idx])
visibility = (baseline - dip) / baseline if baseline > 0 else np.nan

print(f'Baseline matched fourfold rate: {baseline:.3f} counts/s')
print(f'Near-zero-delay matched fourfold rate: {dip:.3f} counts/s at delay={delay[zero_idx]:.0f} ps')
print(f'Estimated HOM visibility: {100 * visibility:.2f}%')


In [ ]:
# Inspect the final aggregated CSV generated by the batch script.
summary[['delay_ps', 'runs', 'total_duration_s', 'bsm_twofold_count_per_s', 'hom_fourfold_count_per_s', 'matched_herald_fourfold_count_per_s', 'mismatched_herald_fourfold_count_per_s']].head(10)


## BSM-Only Results

This section reads `results/hom_bsm_only_config`. In this mode the idlers are not projected or detected; they are routed to passive sinks. The only operational observable plotted here is the BSM-side twofold coincidence rate.

In [ ]:
BSM_ONLY_CONFIG_NAME = 'hom_bsm_only_config'
BSM_ONLY_RESULTS_DIR = EXAMPLE_DIR / 'results' / BSM_ONLY_CONFIG_NAME

def load_bsm_only_csv(name: str) -> pd.DataFrame:
    path = BSM_ONLY_RESULTS_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Run the BSM-only batch config first.')
    df = pd.read_csv(path)
    if 'herald_mode' in df.columns and set(df['herald_mode'].dropna().unique()) != {'bsm_only'}:
        raise ValueError(f'{path} does not look like BSM-only output.')
    return df

bsm_temporal = load_bsm_only_csv('hom_temporal_runs.csv')
bsm_polarization = load_bsm_only_csv('hom_polarization_runs.csv')
bsm_spectral = load_bsm_only_csv('hom_spectral_runs.csv')

print('BSM-only result directory:', BSM_ONLY_RESULTS_DIR.relative_to(REPO_ROOT))
print('temporal rows:', len(bsm_temporal))
print('polarization rows:', len(bsm_polarization))
print('spectral rows:', len(bsm_spectral))


In [ ]:
def summarize_bsm_only(df: pd.DataFrame, x_col: str) -> pd.DataFrame:
    if 'duration_per_run_s' in df.columns:
        duration = df['duration_per_run_s'].astype(float)
    else:
        duration = DURATION_S
    work = df.copy()
    work['bsm_twofold_count_per_s'] = work['bsm_twofold_count'].astype(float) / duration
    out = (
        work.groupby(x_col, sort=True)['bsm_twofold_count_per_s']
        .agg(['mean', 'std', 'count'])
        .reset_index()
        .sort_values(x_col)
    )
    out['std'] = out['std'].fillna(0.0)
    return out

bsm_temporal_summary = summarize_bsm_only(bsm_temporal, 'delay_ps')
bsm_polarization_summary = summarize_bsm_only(bsm_polarization, 'angle_deg')
bsm_spectral_summary = summarize_bsm_only(bsm_spectral, 'detuning_nm')

bsm_temporal_summary.head()


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    bsm_temporal_summary['delay_ps'],
    bsm_temporal_summary['mean'],
    yerr=bsm_temporal_summary['std'],
    marker='o',
    ms=3,
    lw=1.5,
    capsize=2,
    label='BSM twofold',
)
ax.set_xlabel('Relative delay (ps)')
ax.set_ylabel('Counts/s')
ax.set_title('BSM-only temporal scan')
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    bsm_polarization_summary['angle_deg'],
    bsm_polarization_summary['mean'],
    yerr=bsm_polarization_summary['std'],
    marker='o',
    ms=3,
    lw=1.5,
    capsize=2,
    label='BSM twofold',
)
ax.set_xlabel('Waveplate-induced signal rotation (deg)')
ax.set_ylabel('Counts/s')
ax.set_title('BSM-only polarization scan')
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    bsm_spectral_summary['detuning_nm'],
    bsm_spectral_summary['mean'],
    yerr=bsm_spectral_summary['std'],
    marker='o',
    ms=3,
    lw=1.5,
    capsize=2,
    label='BSM twofold',
)
ax.set_xlabel('Signal wavelength detuning (nm)')
ax.set_ylabel('Counts/s')
ax.set_title('BSM-only spectral scan')
ax.legend()
plt.show()


In [ ]:
# Compact table for quick inspection. With runs_per_point=1, std is zero by construction.
print('Temporal BSM-only twofold range:')
print(bsm_temporal_summary[['delay_ps', 'mean', 'std', 'count']].head().to_string(index=False))
print('...')
print(bsm_temporal_summary[['delay_ps', 'mean', 'std', 'count']].tail().to_string(index=False))
